In [3]:
from google.colab import files
uploaded = files.upload()

Saving diabetic_data_cleaned.csv to diabetic_data_cleaned (1).csv


In [4]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data
df = pd.read_csv('diabetic_data_cleaned.csv')

print(f"Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nFirst few rows:")
df.head()

Loaded cleaned dataset: 101,766 rows × 46 columns

First few rows:


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30day
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,41,...,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,59,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,11,...,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,44,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,51,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


In [5]:
# Check data types
print("Data Types:")
print("="*60)
print(df.dtypes.value_counts())
print()

# Identify categorical columns (object type)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}):")
print(categorical_cols)
print()

# Check unique values in each categorical column
print("Unique values in categorical columns:")
print("="*60)
for col in categorical_cols[:10]:  # Show first 10
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} unique values")
    if unique_count <= 10:  # Show values if <= 10 categories
        print(f"  → {df[col].unique()[:10]}")
    print()

Data Types:
object    32
int64     14
Name: count, dtype: int64

Categorical columns (32):
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Unique values in categorical columns:
race: 6 unique values
  → ['Caucasian' 'AfricanAmerican' 'Unknown' 'Other' 'Asian' 'Hispanic']

gender: 3 unique values
  → ['Female' 'Male' 'Unknown/Invalid']

age: 10 unique values
  → ['[0-10)' '[10-20)' '[20-30)' '[30-40)' '[40-50)' '[50-60)' '[60-70)'
 '[70-80)' '[80-90)' '[90-100)']

diag_1: 717 unique values

diag_2: 749 unique values

diag_3: 790 unique values

metformin: 4 uniqu

In [6]:
# Create age mapping (midpoint of each range)
age_mapping = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}

# Apply the mapping
df['age_numeric'] = df['age'].map(age_mapping)

# Verify it worked
print("Age conversion check:")
print("="*60)
print("Original age values:")
print(df['age'].value_counts().sort_index())
print("\nNew numeric age values:")
print(df['age_numeric'].value_counts().sort_index())

# Check for any missing values (shouldn't be any)
print(f"\nMissing values in age_numeric: {df['age_numeric'].isnull().sum()}")

Age conversion check:
Original age values:
age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64

New numeric age values:
age_numeric
5       161
15      691
25     1657
35     3775
45     9685
55    17256
65    22483
75    26068
85    17197
95     2793
Name: count, dtype: int64

Missing values in age_numeric: 0


In [7]:
# Binary encode gender
# Male = 1, Female = 0, Unknown = -1 (or we could drop them)
df['gender_male'] = df['gender'].map({
    'Male': 1,
    'Female': 0,
    'Unknown/Invalid': -1  # Flag as unknown
})

# Verify
print("Gender encoding check:")
print("="*60)
print("Original gender distribution:")
print(df['gender'].value_counts())
print("\nEncoded gender distribution:")
print(df['gender_male'].value_counts())

# Check for missing
print(f"\nMissing values: {df['gender_male'].isnull().sum()}")

Gender encoding check:
Original gender distribution:
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

Encoded gender distribution:
gender_male
 0    54708
 1    47055
-1        3
Name: count, dtype: int64

Missing values: 0


In [8]:
# List of all medication columns
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Medication mapping
med_mapping = {
    'No': 0,
    'Steady': 1,
    'Down': 2,
    'Up': 3
}

# Apply to all medication columns
for col in medication_cols:
    new_col_name = f'{col}_encoded'
    df[new_col_name] = df[col].map(med_mapping)

print("Medication encoding complete!")
print(f"Encoded {len(medication_cols)} medication columns")

# Verify one example
print("\nExample - metformin:")
print("Original:")
print(df['metformin'].value_counts())
print("\nEncoded:")
print(df['metformin_encoded'].value_counts())

Medication encoding complete!
Encoded 23 medication columns

Example - metformin:
Original:
metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: count, dtype: int64

Encoded:
metformin_encoded
0    81778
1    18346
3     1067
2      575
Name: count, dtype: int64


In [9]:
# Check what values they have
print("Checking 'change' column:")
print(df['change'].value_counts())
print()

print("Checking 'diabetesMed' column:")
print(df['diabetesMed'].value_counts())
print()

# Encode them (assuming Yes=1, No=0)
df['change_encoded'] = df['change'].map({'No': 0, 'Ch': 1})
df['diabetesMed_encoded'] = df['diabetesMed'].map({'No': 0, 'Yes': 1})

# Verify
print("Encoded values:")
print("="*60)
print("change_encoded:")
print(df['change_encoded'].value_counts())
print("\ndiabetesMed_encoded:")
print(df['diabetesMed_encoded'].value_counts())

# Check for missing
print(f"\nMissing in change_encoded: {df['change_encoded'].isnull().sum()}")
print(f"Missing in diabetesMed_encoded: {df['diabetesMed_encoded'].isnull().sum()}")

Checking 'change' column:
change
No    54755
Ch    47011
Name: count, dtype: int64

Checking 'diabetesMed' column:
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64

Encoded values:
change_encoded:
change_encoded
0    54755
1    47011
Name: count, dtype: int64

diabetesMed_encoded:
diabetesMed_encoded
1    78363
0    23403
Name: count, dtype: int64

Missing in change_encoded: 0
Missing in diabetesMed_encoded: 0


In [10]:
# One-hot encode race
race_dummies = pd.get_dummies(df['race'], prefix='race')

# Show what was created
print("One-hot encoded race columns:")
print("="*60)
print(race_dummies.head())
print()
print(f"Created {race_dummies.shape[1]} columns:")
print(race_dummies.columns.tolist())

# Add to main dataframe
df = pd.concat([df, race_dummies], axis=1)

print(f"\nDataframe shape after adding race columns: {df.shape}")

One-hot encoded race columns:
   race_AfricanAmerican  race_Asian  race_Caucasian  race_Hispanic  \
0                 False       False            True          False   
1                 False       False            True          False   
2                  True       False           False          False   
3                 False       False            True          False   
4                 False       False            True          False   

   race_Other  race_Unknown  
0       False         False  
1       False         False  
2       False         False  
3       False         False  
4       False         False  

Created 6 columns:
['race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'race_Unknown']

Dataframe shape after adding race columns: (101766, 79)


In [11]:
# Function to group diagnosis codes
def group_diagnosis(code):
    """
    Group ICD-9 diagnosis codes into categories
    """
    if pd.isna(code) or code == 'Unknown':
        return 'Unknown'

    # Convert to numeric (handle V and E codes)
    try:
        # Remove V and E prefixes
        code_str = str(code).replace('V', '').replace('E', '')
        code_num = float(code_str)
    except:
        return 'Other'

    # Group by ICD-9 ranges
    if 390 <= code_num <= 459 or code_num == 785:
        return 'Circulatory'
    elif 460 <= code_num <= 519 or code_num == 786:
        return 'Respiratory'
    elif 520 <= code_num <= 579 or code_num == 787:
        return 'Digestive'
    elif 250 <= code_num < 251:
        return 'Diabetes'
    elif 800 <= code_num <= 999:
        return 'Injury'
    elif 710 <= code_num <= 739:
        return 'Musculoskeletal'
    elif 580 <= code_num <= 629 or code_num == 788:
        return 'Genitourinary'
    elif 140 <= code_num <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# Apply to all three diagnosis columns
df['diag_1_group'] = df['diag_1'].apply(group_diagnosis)
df['diag_2_group'] = df['diag_2'].apply(group_diagnosis)
df['diag_3_group'] = df['diag_3'].apply(group_diagnosis)

# Check the results
print("Diagnosis grouping results:")
print("="*60)
print("\ndiag_1_group distribution:")
print(df['diag_1_group'].value_counts())
print()
print(f"Reduced from {df['diag_1'].nunique()} unique codes to {df['diag_1_group'].nunique()} categories!")

Diagnosis grouping results:

diag_1_group distribution:
diag_1_group
Circulatory        30437
Other              18171
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6975
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown               21
Name: count, dtype: int64

Reduced from 717 unique codes to 10 categories!


In [12]:
# One-hot encode the grouped diagnoses
diag1_dummies = pd.get_dummies(df['diag_1_group'], prefix='diag1')
diag2_dummies = pd.get_dummies(df['diag_2_group'], prefix='diag2')
diag3_dummies = pd.get_dummies(df['diag_3_group'], prefix='diag3')

# Add to main dataframe
df = pd.concat([df, diag1_dummies, diag2_dummies, diag3_dummies], axis=1)

print("Diagnosis encoding complete!")
print("="*60)
print(f"Added {diag1_dummies.shape[1]} diag_1 columns")
print(f"Added {diag2_dummies.shape[1]} diag_2 columns")
print(f"Added {diag3_dummies.shape[1]} diag_3 columns")
print(f"\nNew dataframe shape: {df.shape}")

Diagnosis encoding complete!
Added 10 diag_1 columns
Added 10 diag_2 columns
Added 10 diag_3 columns

New dataframe shape: (101766, 112)


In [13]:
# Feature 1: Total hospital encounters in past year
df['total_visits'] = (df['number_emergency'] +
                      df['number_inpatient'] +
                      df['number_outpatient'])

# Feature 2: Polypharmacy flag (15+ medications)
df['polypharmacy'] = (df['num_medications'] >= 15).astype(int)

# Feature 3: High-risk age group (50-80 years - from our exploration)
df['high_risk_age'] = ((df['age_numeric'] >= 50) & (df['age_numeric'] <= 80)).astype(int)

# Verify
print("New features created:")
print("="*60)
print("\nTotal visits distribution:")
print(df['total_visits'].describe())
print("\nPolypharmacy flag:")
print(df['polypharmacy'].value_counts())
print("\nHigh-risk age group:")
print(df['high_risk_age'].value_counts())
print(f"\nFinal dataframe shape: {df.shape}")

New features created:

Total visits distribution:
count    101766.000000
mean          1.202759
std           2.291781
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          80.000000
Name: total_visits, dtype: float64

Polypharmacy flag:
polypharmacy
1    52313
0    49453
Name: count, dtype: int64

High-risk age group:
high_risk_age
1    65807
0    35959
Name: count, dtype: int64

Final dataframe shape: (101766, 115)


In [14]:
# Columns to drop (original text versions we've encoded)
columns_to_drop = [
    # Demographics (encoded versions exist)
    'race', 'gender', 'age',

    # Diagnoses (grouped and encoded)
    'diag_1', 'diag_2', 'diag_3',
    'diag_1_group', 'diag_2_group', 'diag_3_group',

    # Medications (encoded versions exist)
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone',

    # Other (encoded versions exist)
    'change', 'diabetesMed', 'readmitted'
]

# Drop them
df_final = df.drop(columns=columns_to_drop)

print("Dropped original categorical columns")
print("="*60)
print(f"Shape before: {df.shape}")
print(f"Shape after: {df_final.shape}")
print(f"Columns removed: {len(columns_to_drop)}")

Dropped original categorical columns
Shape before: (101766, 115)
Shape after: (101766, 80)
Columns removed: 35


In [15]:
# Save the model-ready dataset
df_final.to_csv('diabetic_data_model_ready.csv', index=False)

print("="*60)
print("FEATURE ENGINEERING COMPLETE!")
print("="*60)
print(f"\nFinal dataset shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print(f"\nAll columns are now numeric and ready for machine learning!")
print(f"\nTarget variable: readmitted_30day")
print(f"  - Class 0 (not readmitted <30): {(df_final['readmitted_30day']==0).sum():,}")
print(f"  - Class 1 (readmitted <30): {(df_final['readmitted_30day']==1).sum():,}")
print(f"\n Saved as: diabetic_data_model_ready.csv")
print("="*60)

# Quick check - all columns numeric?
print("\nData types check:")
print(df_final.dtypes.value_counts())

FEATURE ENGINEERING COMPLETE!

Final dataset shape: 101,766 rows × 80 columns

All columns are now numeric and ready for machine learning!

Target variable: readmitted_30day
  - Class 0 (not readmitted <30): 90,409
  - Class 1 (readmitted <30): 11,357

 Saved as: diabetic_data_model_ready.csv

Data types check:
int64    44
bool     36
Name: count, dtype: int64


In [16]:
# Download the file
from google.colab import files
files.download('diabetic_data_model_ready.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>